# Harness Engineering: OpenCode Case Study
## Chapter 1: Overall Harness

In earlier labs, you built an agent loop and connected it to tools. Now we will use **OpenCode**, an open-source coding agent, to ask a broader question: what must we put around that loop to give it a clear task, control its actions, and check its work?

We will follow one small repair. A program averages three grades: 80, 100, and a missing score. The requirement says to ignore missing scores, so the answer should be 90. The program returns 60. As OpenCode works, we will examine what the model requests, how OpenCode carries it out, and how we check the result.

Follow five questions:
1. Where is the bug?
2. What does OpenCode actually do?
3. How does a tool result help it choose the next step?
4. Did it really fix the program?
5. How can a later conversation continue the same task?

**Part 1** follows the repair. **Part 2** explains the harness around the model. **Part 3** traces the key steps in real source code.


### Part 1: Follow the Repair
#### 1. Where is the bug?

Start with the project, before asking a model to change it. Run the setup cell in a fresh Colab runtime. It installs OpenCode `1.18.34` and makes a new copy of the example; earlier copies are left untouched. No model is called yet.

The directory below shows what each file is for. Read the requirements, program, data, and test before running them.


In [ ]:
#@title Prepare OpenCode and the example project
from pathlib import Path
import subprocess
import sys
import tempfile

REPO_URL = "https://github.com/SleepyLGod/AIST4020-Lab-Tutorials.git"
LAB_RELATIVE = Path("labs/opencode/overall-harness")
LAB_DIR = next((p for p in (Path.cwd(), Path.cwd() / LAB_RELATIVE)
                if (p / "lab_support.py").is_file()), None)
if LAB_DIR is None:
    course = Path(tempfile.mkdtemp(prefix="aist4020-course-"))
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(course)], check=True)
    LAB_DIR = course / LAB_RELATIVE
sys.path.insert(0, str(LAB_DIR))
import lab_support as support
if Path(support.__file__).resolve() != (LAB_DIR / "lab_support.py").resolve():
    raise RuntimeError("Restart the runtime to load this lab's helper.")
lab = support.open_lab(LAB_DIR, version="1.18.34")


In [ ]:
support.show_project(lab.project)
for name in ("README.md", "grades.py", "scores.csv", "test_grades.py"):
    support.show_details(name, (lab.project / name).read_text(), opened=True)


Run the program and its test yourself with `lab.check(...)`, which executes a command without asking the model. Compare the output with the required value of 90.

The `assert` lines confirm that we reproduced the bug: the test failed and reported `60.0 != 90.0`. Finally, `snapshot(...)` records the current files in `before`, so we can see what OpenCode changes later.


In [ ]:
initial_average = lab.check([
    sys.executable, "-c",
    "from grades import average; print('Average:', average('scores.csv'))",
])
TEST_COMMAND = ["python3", "-m", "unittest", "-v"]
initial_test = lab.check(TEST_COMMAND)
assert not initial_test.timed_out and initial_test.returncode != 0
assert "60.0 != 90.0" in initial_test.stdout + initial_test.stderr
before = support.snapshot(lab.project)


In `grades.py`, `scores` contains two numbers, but `len(rows)` counts all three rows. That makes the calculation 180 / 3. The requirement calls for counting only the recorded scores.

#### 2. What does OpenCode actually do?

**Choose a model.**

Now let OpenCode work on the same problem. Choose one backend below and set `ENABLE_MODEL_CALLS = True`. Keep `deepseek`, or select `nvidia` and enter your key when prompted by the next setup cell. You can also save the key in Colab Secrets. For `ollama`, no key is needed; use a GPU runtime. API calls may use paid credit, and one repair can make several model calls.

`base_url` is the model service address, and `model` selects the model. `key_env` names the environment variable that holds your key; do not replace it with the key itself. To use your own provider, fill in the `custom` entry and select it.


In [ ]:
BACKENDS = {
    "deepseek": dict(base_url="https://api.deepseek.com/v1",
                     model="deepseek-flash", key_env="DEEPSEEK_API_KEY"),
    "nvidia": dict(base_url="https://integrate.api.nvidia.com/v1",
                   model="z-ai/glm-5.3", key_env="NVIDIA_NIM_API_KEY"),
    "ollama": dict(base_url="http://127.0.0.1:11434/v1",
                   model="qwen3.5:9b-q4_K_M", key_env=""),
    "custom": dict(base_url="", model="", key_env=""),
}

backend = "deepseek"
ENABLE_MODEL_CALLS = False


**Set permissions for this repair.**

We have chosen the permissions below for this exercise. OpenCode needs to inspect the project, fix `grades.py`, and test it; it does not need to change the test file. Our configuration allows:

- Reading files (`read`), finding files by name (`glob`), and searching their contents (`grep`).
- Editing `grades.py`, but not the other project files (`edit`).
- Running `python3 -m unittest -v` in the terminal (`bash`).

`"*": "deny"` rejects other tool actions unless explicitly allowed; `external_directory` rejects requests to access files outside the project. These are tool rules, not an isolated computer.

Run the cell to save the selected model and these rules with `lab.configure(...)`. Check the printed model and permissions. No repair starts yet. If you chose Ollama, the first setup also downloads and starts the model, so allow time for it to finish.


In [ ]:
PERMISSIONS = {
    "*": "deny",
    "read": "allow",
    "glob": "allow",
    "grep": "allow",
    "edit": {"*": "deny", "grades.py": "allow", "**/grades.py": "allow"},
    "bash": {"*": "deny", "python3 -m unittest -v": "allow"},
    "external_directory": "deny",
}
lab.configure(backend, BACKENDS, permissions=PERMISSIONS, enabled=ENABLE_MODEL_CALLS)


**Check the connection.**

Ask for a greeting before sending the repair task. This request cannot use tools. Look for a text reply; resolve any connection error before moving on.

`lab.run(...)` is our course helper, not an OpenCode Python API. With the default model, it runs:

```bash
opencode run --pure --format json --agent build --model course/deepseek-flash \
  "Reply with one sentence: OpenCode is ready for the lab. Do not use tools."
```

`--agent build` selects OpenCode's built-in agent for tasks such as editing code; `--model` selects the language model it uses. The helper prints the actual command and working folder before each run.


In [ ]:
hello = lab.run(
    "Reply with one sentence: OpenCode is ready for the lab. Do not use tools.",
    tools=False, label="hello",
)


**Start the repair.**

The task below asks OpenCode to test, fix, and test again, then put the test result first in its report. We will return to that report-order instruction later.

Run it and follow the actions. Which file does it read first? Does it run the test before changing the code?

`repair` holds this run's reply, tool actions, and session ID, which we inspect below. The helper stops a run after five minutes.


In [ ]:
repair_prompt = """Read README.md and grades.py. Run python3 -m unittest -v to reproduce
the bug. Fix only grades.py, then rerun the same test command.
Report what changed and what the test actually returned.
For this session, put the test result before the explanation of the fix."""

repair = lab.run(repair_prompt)
print("Repair session:", repair.session_id or "not reported")


#### 3. How does a tool result help it choose the next step?

The table lists the actions reported in your run, in order. Find a test command, expand its output, and look at the next action. Did a failed test lead to an edit? Was there another test after that edit?

If your run stopped early, use its last action or error to see where it stopped.


In [ ]:
support.show_tool_actions(repair)


If a test returns `60.0 != 90.0`, the program produced 60 where the test expected 90. The model can use this mismatch when deciding what to change. `completed` means the tool finished, not that the test passed.


#### 4. Did it really fix the program?

Compare the files with the `before` snapshot. The **diff** shows the lines that changed. Then rerun the test and check the average directly, without relying on the test file.

Read the before-and-after table: does the average now equal 90, does the test pass, and did only `grades.py` change?

These are your checks, not the agent's. Use the action table above to see whether the agent also retested.


In [ ]:
changed_files = support.show_changes(before, support.snapshot(lab.project), lab.redact)

test_after = lab.check(TEST_COMMAND)

# Check the expected value without relying on the project's test file.
average_check = lab.check([
    sys.executable, "-c",
    "from grades import average; actual = average('scores.csv'); "
    "print('Average:', actual); assert actual == 90.0, actual",
])
support.show_repair_comparison(
    initial_average, initial_test, average_check, test_after, changed_files, lab.redact,
)


Read the final answer alongside the actions from Question 3. If it says the tests passed, find the test **after the last edit**. If only your own test passed, the program may be fixed even though the agent skipped that requested step.

You now have three things to compare: the changed code, the test results, and what OpenCode said it did.


#### 5. How can a later conversation continue the same task?

Now continue the conversation about the repair. Ask, "What report order did I request?" once in the repair session and once in a new session. A **session** is one saved conversation. The model and project stay the same.

Passing `session_id` adds `--session` to the command; leaving it out starts a new conversation. Here, `tools=False` selects a configuration that denies tool use, so neither request can look for the answer in files.


In [ ]:
if not repair.session_id:
    raise RuntimeError("The repair returned no session ID. Check its error before continuing.")

history_question = """What order did I ask you to use in your report earlier in this session?
If you cannot find an earlier request in this conversation, say so.
Do not read or change project files."""

same = lab.run(history_question, session_id=repair.session_id, tools=False, label="same")
fresh = lab.run(history_question, tools=False, label="fresh")
assert same.session_id == repair.session_id
assert fresh.session_id and fresh.session_id != same.session_id


Compare the two replies side by side below. Does the continued session recall your requested order? How does the new session answer?

The last row checks whether your original instruction appears in the saved user messages. Expand those messages to read it in context; a correct reply alone does not show where the answer came from.


In [ ]:
continued_history = lab.export(same.session_id, label="continued", show=False)
fresh_history = lab.export(fresh.session_id, label="fresh", show=False)

instruction = "For this session, put the test result before the explanation of the fix."
support.show_session_comparison(
    same, fresh, continued_history=continued_history, fresh_history=fresh_history,
    instruction=instruction, redact=lab.redact,
)
assert support.has_user_message(continued_history, instruction)
assert not support.has_user_message(fresh_history, instruction)


### Part 2: What Is the Harness?

The repair depends on more than choosing the next tool. Someone must define the right result, limit what can change, keep the task history, and check the finished work.

> A harness is the setup around the model that helps an agent do a task more reliably: files, instructions, tools, action rules, session history, memory notes, checks, and feedback loops.

Here, we supply the project, task, permission settings, and independent checks. OpenCode prepares model input, runs tools under those rules, and saves the conversation. This example uses session history, without an additional memory system.


The loop is one part of the arrangement below. The final checks are ours; OpenCode does not perform them on our behalf.

![Overall Harness for the repair: we prepare the task, files and rules; OpenCode connects the model to tools and saved history; we check the result and use what we learn to improve a later run.](https://raw.githubusercontent.com/SleepyLGod/AIST4020-Lab-Tutorials/main/labs/opencode/overall-harness/assets/agent-loop.png)

#### Workspace and Instructions

`README.md` says to ignore missing scores, and the test expresses that requirement as an expected value of 90. These files give the repair a goal beyond "make the error disappear." The project folder is the **workspace**. Its files reach the model when a tool reads them; displaying them in this notebook does not send them to the model.

#### Tools and Permissions

Our request tells the model to edit only `grades.py`. The permission settings let OpenCode check proposed actions against that restriction. Tools provide the actual file access and test execution. Instructions describe what we want; permissions govern what those tools may do.

#### Session and Context

If the agent runs the test, its output joins the saved conversation, so the next model call can use it. The report-order instruction is saved there too, which lets a later turn continue the task. The **session** is this saved conversation; **context** is the input assembled for one model call, including instructions, tool descriptions, and relevant conversation history.

#### Checks

A final answer is a claim about the repair. Our test checks the value, the diff shows what changed, and the action record shows whether the agent retested. Together, they let us compare the work with the original request.


#### Feedback: Improve the Next Run

During a run, a failed test can guide the next edit. After a run, we can use the same evidence to improve the setup.

For discussion, suppose the agent requests `python3 -m unittest` and is denied because our rule allows only `python3 -m unittest -v`. Both run the same tests here; `-v` adds more detailed output. We could allow that exact alternative while keeping other commands denied. A later run from a fresh project could show whether this helps the agent retest.

This is harness engineering: change a relevant part of the setup, then check whether it improves the work. The model can stay the same.

Further reading: [Birgitta Böckeler on harness engineering](https://martinfowler.com/articles/harness-engineering.html) and [OpenAI's project example](https://openai.com/index/harness-engineering/).


### Part 3: Find the Same Steps in the Source

Now trace OpenCode's part of the harness: preparing input, checking and executing actions, saving results, and using history in later calls. The task, project files, and independent checks are the course code you ran in Part 1.

Sources use [OpenCode v1.18.34](https://github.com/anomalyco/opencode/tree/v1.18.34), matching the lab. `// ...` omits unrelated lines. In this TypeScript code, read `yield*` as "perform this operation and get its result"; focus on the values passed between calls.


#### 1. Prepare the model's input

Your repair request enters OpenCode with the selected model and session:

```ts
const model = pick(args.model)
const result = await client.session.prompt({
  sessionID,
  agent,
  model,
// ...
  parts: [...files, { type: "text", text: message }],
})
```

*Source: [cli/cmd/run.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/cli/cmd/run.ts#L863)*

`message` is the task you wrote; `sessionID` identifies its conversation. `files` contains explicitly attached files. We attach none here; the agent reads project files through tools.

For the default DeepSeek setup, OpenCode loads its system instructions from `session/prompt/default.txt`, including:

```text
You are opencode, an interactive CLI tool that helps users with software engineering tasks. Use the instructions below and the tools available to you to assist the user.
...
# Doing tasks
...
- Use the available search tools to understand the codebase and the user's query. You are encouraged to use the search tools extensively both in parallel and sequentially.
- Implement the solution using all tools available to you
- Verify the solution if possible with tests. NEVER assume specific test framework or test script. Check the README or search codebase to determine the testing approach.
...
```

*Source: [session/prompt/default.txt](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/prompt/default.txt)*

The instructions tell the model to understand the project, make the change, and check it using the project's tests. Your repair request supplies the specific task. OpenCode assembles the system instructions here:

```ts
const system = [
  [
    ...(input.agent.prompt ? [input.agent.prompt] : SystemPrompt.provider(input.model)),
    ...input.system,
    ...(input.user.system ? [input.user.system] : []),
  ]
    .filter((x) => x)
    .join("\n"),
]
```

*Source: [session/llm/request.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/llm/request.ts#L58)*

The first line selects agent-specific instructions or OpenCode's default instructions for that model; the rest adds further guidance. OpenCode [places these system messages before the conversation](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/llm/request.ts#L101) in `prepared.messages`. Your task stays a user message. The model receives both, plus the available tools:

```ts
result: streamText({
// ...
  tools: prepared.tools,
// ...
  messages: prepared.messages,
  model: wrapLanguageModel({
    model: language,
// ...
  }),
// ...
}),
```

*Source: [session/llm.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/llm.ts#L280)*

What does the model see about a tool? Consider `read`.


#### 2. Turn a request into an action

The model receives a description of `read` and the parameters it accepts:

```text
Read a file or directory from the local filesystem.
...
- The filePath parameter should be an absolute path.
```

*Source: [tool/read.txt](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/read.txt)*

OpenCode converts this definition into a description of the arguments the model must supply:

```ts
export const Parameters = Schema.Struct({
  filePath: Schema.String.annotate({ description: "The absolute path to the file or directory to read" }),
  // ...
})
```

*Source: [tool/read.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/read.ts#L28)*

OpenCode joins the description, argument definition, and execution function in one tool entry:

```ts
const schema = ProviderTransform.schema(input.model, ToolJsonSchema.fromTool(item))
tools[item.id] = tool({
  description: item.description,
  inputSchema: jsonSchema(schema),
  execute(args, options) {
    return run.promise(
      Effect.gen(function* () {
        const ctx = context(args, options)
        // ...
        const result = yield* item.execute(args, ctx)
        const output = {
          ...result,
          // ...
        }
        // ...
        return output
      }),
    )
  },
})
```

*Source: [session/tools.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/tools.ts#L98)*

`item` is a tool such as `read`. The model receives its `description` and `inputSchema` (the accepted arguments); `execute` stays in OpenCode. To read `grades.py`, the model supplies its absolute path as `filePath`. OpenCode passes those arguments (`args`) to `item.execute` and returns its output.

`ctx` provides session information and permission checks. Inside `read`, permission is checked before the file is opened:

```ts
yield* ctx.ask({
  permission: "read",
  patterns: [path.relative(instance.worktree, filepath)],
// ...
})
// ...
const file = yield* lines(filepath, { limit: params.limit ?? DEFAULT_READ_LIMIT, offset: params.offset || 1 })
// ...
return {
  title,
  output,
// ...
}
```

*Source: [tool/read.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/read.ts#L255)*

If `ctx.ask` allows the read, `lines(...)` reads the file and `output` carries its contents back. The `bash` tool runs our test command instead; its result follows the same path back to the conversation.


#### 3. Keep the test result with its request

Suppose the agent's test returns `60.0 != 90.0`. Follow that result from storage to the next model call. First, OpenCode completes the tool record:

```ts
case "tool-result": {
// ...
  yield* completeToolCall(value.id, output)
  return
}
```

*Source: [session/processor.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/processor.ts#L383)*

A saved message can contain text and tool records. Inside `completeToolCall`, `match.part` is the record for this command; its `state` holds the status, arguments, and result. OpenCode updates that record when the command finishes:

```ts
const match = yield* readToolCall(toolCallID)
if (!match || match.part.state.status !== "running") return
yield* session.updatePart({
  ...match.part,
  state: {
    status: "completed",
    input: match.part.state.input,
    output: output.output,
// ...
  },
})
```

*Source: [session/processor.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/processor.ts#L169)*

The outer `output` is the returned object; its `output` field contains the test report, including `60.0 != 90.0` in this example. `toolCallID` ties it to the test command. The record becomes `completed` because the command finished.

OpenCode also writes these records to its database. You can follow that storage code in [session/session.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/session.ts#L637) and [session/projector.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/core/src/session/projector.ts#L310).


#### 4. Use the result in the next model call

The next loop pass loads the same session, bringing the saved test report back into the conversation:

```ts
let msgs = yield* MessageV2.filterCompactedEffect(sessionID).pipe(
  Effect.provideService(Database.Service, database),
)
```

*Source: [session/prompt.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/prompt.ts#L1092)*

`msgs` holds that conversation, including the saved failure in our example. The function also handles shortened history.

This is also why continuing a session matters: the same `sessionID` lets OpenCode load your earlier request to put the test result first. A new session does not load that conversation.

OpenCode then converts `msgs` into messages for the model, stored as `modelMsgs`:

```ts
MessageV2.toModelMessagesEffect(msgs, model),
```

*Source: [session/prompt.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/prompt.ts#L1262)*

For the completed test command, the conversion includes its returned report:

```ts
if (part.state.status === "completed") {
// ...
  assistantMessage.parts.push({
    type: ("tool-" + part.tool) as `tool-${string}`,
    state: "output-available",
    toolCallId: part.callID,
    input: part.state.input,
    output,
// ...
  })
}
```

*Source: [session/message-v2.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/message-v2.ts#L296)*

Inside `assistantMessage.parts`, this is a tool result, not text written by the model. `output` carries the test report, and `toolCallId` links it to the command. OpenCode converts this record into model messages, so the report reaches the next model step through `modelMsgs`:

```ts
const result = yield* handle.process({
// ...
  system,
  messages: [
    ...modelMsgs,
// ...
  ],
  tools,
  model,
// ...
})
```

*Source: [session/prompt.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/prompt.ts#L1272)*

**`60.0 != 90.0` → saved tool result → loaded conversation → next model input.** The model can use the reported mismatch to decide what to change.

OpenCode's CLI reports finished or failed tool actions as JSON. The notebook reads those records to display each request and its result. [Source: CLI tool events](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/cli/cmd/run.ts#L724).


#### What to remember

The model proposes the next action. OpenCode connects that request to tools, checks the action, and returns the result through the conversation. The harness includes this loop and the files, instructions, rules, state, and checks around it.

#### References

- [OpenCode CLI](https://opencode.ai/docs/cli/), [rules](https://opencode.ai/docs/rules/), and [permissions](https://opencode.ai/docs/permissions/).
- [OpenCode v1.18.34 source](https://github.com/anomalyco/opencode/tree/v1.18.34).
- [Birgitta Böckeler: Harness engineering for coding agent users](https://martinfowler.com/articles/harness-engineering.html).
- [OpenAI: Harness engineering](https://openai.com/index/harness-engineering/).

#### Optional: Keep your run

The helper saves command output and exported conversations. To download them, run the cell below. Known keys are masked; review file contents and personal text before sharing.

When finished, use **Runtime → Disconnect and delete runtime** to release the Colab machine. Download first if you want to keep the records.


In [ ]:
DOWNLOAD_RECORDS = False
if DOWNLOAD_RECORDS:
    lab.save_record({
        "repair_session": repair.session_id,
        "fresh_session": fresh.session_id,
        "changed_files": changed_files,
        "your_test_passed": test_after.returncode == 0 and not test_after.timed_out,
        "average_is_90": average_check.returncode == 0 and not average_check.timed_out,
    })
    lab.download_records()
